In [1]:
import os
import pandas as pd
import numpy as np
import json
import re

from sklearn.metrics import f1_score, accuracy_score

from sentence_transformers import SentenceTransformer
from rouge import Rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from scipy.spatial.distance import cosine
from typing import Dict, List 

/orcd/software/core/001/pkg/miniforge/25.11.0-0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ground_truth=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/BIDS_data/anotated_processed.csv")
results_1_ovis=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_1/ovis2/20260113_1449/predictions.csv")
results_1_qwen=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_1/qwen25/20260113_1943/predictions.csv")
results_2_ovis=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_2/ovis2/20260113_1449/predictions.csv")
results_2_qwen=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_2/qwen25/20260113_1527/predictions.csv")
results_3_ovis=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_3/ovis2/20260113_1450/predictions.csv")
results_3_qwen=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_3/qwen25/20260113_1812/predictions.csv")
results_1_2_ovis=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_1/ovis2/20260114_1741/predictions.csv")
results_1_2_qwen=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_1/qwen25/20260114_1741/predictions.csv")
results_2_2_ovis=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_2/ovis2/20260114_1742/predictions.csv")
results_2_2_qwen=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_2/qwen25/20260114_1741/predictions.csv")
results_3_2_ovis=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_3/ovis2/20260114_1742/predictions.csv")
results_3_2_qwen=pd.read_csv("/orcd/scratch/bcs/001/sensein/sails/pred2annot_evaluation/all_annotations_3/qwen25/20260114_1741/predictions.csv")

In [3]:
# -----------------------
# 1) Normalization (keeps "na"/"n_a"/"n/a" as a real label -> "na")
# -----------------------
MISSING_STRINGS = {"", "none", "null"}
NA_LABEL_STRINGS = {"na", "n/a", "n_a"}

def norm_text(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()

    # treat n/a variants as a meaningful label (not missing)
    if s in NA_LABEL_STRINGS:
        return "na"

    if s in MISSING_STRINGS:
        return pd.NA

    # unify underscores/spaces like before
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# -----------------------
# 2) Parse JSON that may be wrapped in ```json ... ```
# -----------------------
CODEBLOCK_RE = re.compile(r"^```(?:json)?\s*(.*?)\s*```$", re.DOTALL | re.IGNORECASE)
_num_re = re.compile(r"[-+]?\d*\.?\d+")
def norm_num(x):
    """Extract the first number from x and return it as an int (or pd.NA)."""
    if x is None or x is pd.NA:
        return pd.NA
    # pandas/np NaN
    if isinstance(x, float) and np.isnan(x):
        return pd.NA

    # already numeric
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return pd.NA if np.isnan(x) else int(round(float(x)))

    s = str(x).strip()
    if not s:
        return pd.NA

    m = _num_re.search(s)
    if not m:
        return pd.NA

    # "3.0" -> 3, "3.7" -> 4 (because round); change to int(float(...)) if you prefer truncation
    return int(round(float(m.group(0))))
def safe_load(s):
    # Always return a dict
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return {}

    # if it's already parsed
    if isinstance(s, dict):
        return s
    if isinstance(s, list):
        # if it's a list of dicts, take first; otherwise treat as empty
        return s[0] if (len(s) > 0 and isinstance(s[0], dict)) else {}

    txt = str(s).strip()

    # remove ```json ... ``` fences
    m = CODEBLOCK_RE.match(txt)
    if m:
        txt = m.group(1).strip()

    # remove stray "json" prefix if present
    txt = re.sub(r"^\s*json\s*", "", txt, flags=re.IGNORECASE).strip()

    try:
        obj = json.loads(txt)
    except Exception:
        return {}

    # json.loads might return list; normalize to dict
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, list):
        return obj[0] if (len(obj) > 0 and isinstance(obj[0], dict)) else {}

    return {}

# -----------------------
# 3) Build prediction table from results
# -----------------------
def analyse_df(results_df,GT_COLS,PRED_TO_GT={},TEXT_COLS={},NUM_COLS={}):
    pred = pd.json_normalize(results_df["raw_prediction"].map(safe_load))
    pred["video_path"] = results_df["video_path"].values

    print(PRED_TO_GT)
    pred = pred.rename(columns=PRED_TO_GT)
    pred = pred[["video_path"] + GT_COLS].copy()
    # ensure all expected cols exist
    for c in GT_COLS:
        if c not in pred.columns:
            pred[c] = pd.NA
    # normalize predictions
    for c in GT_COLS:
        pred[c] = pred[c].map(norm_text)

    for c in TEXT_COLS:
        pred[c] = pred[c].map(norm_text)
    for c in NUM_COLS:
        pred[c] = pred[c].map(norm_num).astype("Int64")


    locomotion_map = {
        "crawling": "crawl",
        "cruising": "cruise",
        "walking": "walk",
        "running": "run",
        "multiple": "multiple",
        "vehicle": "vehicle",
    }
    if "Locomotion_type" in pred.columns:
        pred["Locomotion_type"] = (
            pred["Locomotion_type"]
              .astype("string")
              .str.strip()
              .str.lower()
              .str.replace("_", " ", regex=False)   # so n_a -> n a
              .replace(locomotion_map)
              .replace({"n a": pd.NA, "na": pd.NA, "n/a": pd.NA})  # make n_a become missing for this field
        )
    if "Support_type" in pred.columns:
        # fix the weird value in predictions
        pred["Support_type"] = (
            pred["Support_type"]
              .astype("string")
              .str.strip()
              .replace({"verbal and physical": "both"})
        )
    # -----------------------
    # 4) Prepare ground truth (assumes column names already match)

    # -----------------------
    GT_PATH_COL = "BidsProcessed"   # change if ground truth path column is different for this task
    
    gt = ground_truth.copy()
    for c in GT_COLS:
        if c not in gt.columns:
            raise KeyError(f"ground_truth is missing column: {c}")
        gt[c] = gt[c].map(norm_text)
    
    for c in TEXT_COLS:
        gt[c] = gt[c].map(norm_text)
    for c in NUM_COLS:
        gt[c] = gt[c].map(norm_num).astype("Int64")

    # -----------------------
    # 5) Merge
    # -----------------------

    merged = pred.merge(
        gt[[GT_PATH_COL] + GT_COLS],
        left_on="video_path",
        right_on=GT_PATH_COL,
        how="left",
        suffixes=("_pred", "_gt"),
        indicator=True
    )
    
    # -----------------------
    # 6) Match columns + Accuracy (only where GT exists)
    # -----------------------
    for c in GT_COLS:
        both_na = merged[f"{c}_pred"].isna() & merged[f"{c}_gt"].isna()
        merged[f"{c}_match"] = (merged[f"{c}_pred"] == merged[f"{c}_gt"]) | both_na
    
    rows = []
    for c in GT_COLS:
        mask_eval = (merged["_merge"] == "both") & merged[f"{c}_gt"].notna()
        n_eval = int(mask_eval.sum())
        n_correct = int(merged.loc[mask_eval, f"{c}_match"].sum())
        acc = (n_correct / n_eval) if n_eval else np.nan

        rows.append({
            "annotation": c,
            "n_eval": n_eval,
            "n_correct": n_correct,
            "accuracy": acc,
        })

    metrics_df = (
        pd.DataFrame(rows)
          .sort_values(["accuracy", "n_eval"], ascending=[False, False], na_position="last")
          .reset_index(drop=True)
    )
    return pred, gt, merged,metrics_df
    
    # -----------------------
    # 7) Macro/Weighted F1 for categorical fields
    # -----------------------
def f1_metrics(df, col):
    mask = (df["_merge"] == "both") & df[f"{col}_gt"].notna()
    if mask.sum() == 0:
        return {"n": 0, "accuracy": np.nan, "macro_f1": np.nan, "weighted_f1": np.nan}

    y_true = df.loc[mask, f"{col}_gt"].astype("string")
    y_pred = df.loc[mask, f"{col}_pred"].astype("string").fillna("__MISSING__")

    return {
        "n": int(mask.sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }




In [4]:
def compute_f1_metrics(
    merged: pd.DataFrame,
    col: str,
    *,
    positive: str = "yes",
) -> dict:
    y_true = merged[f"{col}_gt"]
    y_pred = merged[f"{col}_pred"]

    # Evaluate only where GT exists for that field
    mask = y_true.notna()
    n = int(mask.sum())
    if n == 0:
        return {
            "n": 0,
            "accuracy": np.nan,
            "macro_f1": np.nan,
            "weighted_f1": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "tp": 0, "fp": 0, "tn": 0, "fn": 0,
        }

    yt = y_true[mask].astype("string").str.strip().str.lower()
    yp = y_pred[mask].astype("string").str.strip().str.lower()

    pos = str(positive).strip().lower()

    # Binary masks for confusion matrix (treat anything != pos as negative)
    yt_pos = yt.eq(pos)
    yp_pos = yp.eq(pos)

    tp = int((yt_pos & yp_pos).sum())
    fp = int((~yt_pos & yp_pos).sum())
    tn = int((~yt_pos & ~yp_pos).sum())
    fn = int((yt_pos & ~yp_pos).sum())

    def safe_div(num, den):
        return float(num / den) if den else np.nan

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)          # same as sensitivity
    sensitivity = recall
    specificity = safe_div(tn, tn + fp)

    return {
        "n": n,
        "accuracy": float(accuracy_score(yt, yp)),
        "macro_f1": float(f1_score(yt, yp, average="macro")),
        "weighted_f1": float(f1_score(yt, yp, average="weighted")),
        "precision": precision,
        "recall": recall,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
    }



In [5]:
class Evaluator:
    """
    Computes lexical metrics and semantic similarity metrics 
    Attributes:
        rouge: ROUGE metric calculator.
        embedding_model: SentenceTransformer for semantic similarity.
        embedding_model_name: Name of the embedding model.
    """
    
    def __init__(self, embedding_model_name: str = 'all-MiniLM-L6-v2'):
        """Initializes evaluator with embedding model.
        
        Args:
            embedding_model_name: Name of sentence-transformers model.
                Options:
                - 'all-MiniLM-L6-v2': Fast, 384-dim (default)
                - 'all-mpnet-base-v2': Higher quality, 768-dim
                - 'paraphrase-MiniLM-L6-v2': Paraphrase detection
        """
        self.rouge = Rouge()
        print(f"Loading embedding model: {embedding_model_name}")
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.embedding_model_name = embedding_model_name
    
    def _compute_word_overlap(self, pred: str, truth: str) -> float:
        """Computes word overlap ratio between two texts.
        
        Args:
            pred: Predicted text.
            truth: Ground truth text.
            
        Returns:
            Overlap ratio (0.0 to 1.0).
        """
        pred_words = set(pred.lower().split())
        truth_words = set(truth.lower().split())
        
        if not truth_words:
            return 0.0
        
        overlap = len(pred_words & truth_words)
        return overlap / len(truth_words)
    
    def _compute_bleu(self, pred: str, truth: str) -> float:
        """Computes BLEU score with smoothing.
        
        Args:
            pred: Predicted text.
            truth: Ground truth text.
            
        Returns:
            BLEU score (0.0 to 1.0).
        """
        reference = [truth.split()]
        candidate = pred.split()
        smoothing = SmoothingFunction().method1
        
        try:
            return sentence_bleu(
                reference, 
                candidate, 
                smoothing_function=smoothing
            )
        except:
            return 0.0
    
    def _compute_rouge(self, pred: str, truth: str) -> Dict[str, float]:
        """Computes ROUGE scores.
        
        Args:
            pred: Predicted text.
            truth: Ground truth text.
            
        Returns:
            Dictionary with rouge-1, rouge-2, rouge-l F1 scores.
        """
        try:
            scores = self.rouge.get_scores(pred, truth)[0]
            return {
                'rouge1': scores['rouge-1']['f'],
                'rouge2': scores['rouge-2']['f'],
                'rougeL': scores['rouge-l']['f']
            }
        except:
            return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    
    def _compute_semantic_similarity(
        self, 
        predictions: List[str], 
        ground_truths: List[str]
    ) -> Dict[str, float]:
        """Computes semantic similarity metrics using embeddings.
        
        Args:
            predictions: List of predicted texts.
            ground_truths: List of ground truth texts.
            
        Returns:
            Dictionary with cosine similarity, euclidean distance, and dot product.
        """
        # Filter valid pairs
        valid_pairs = [
            (str(pred), str(truth))
            for pred, truth in zip(predictions, ground_truths)
            if pred and truth and not pd.isna(pred) and not pd.isna(truth)
        ]
        
        if not valid_pairs:
            return {
                'cosine_similarity': 0.0,
                'euclidean_distance': 0.0,
                'dot_product': 0.0,
                'count': 0
            }
        
        pred_texts, truth_texts = zip(*valid_pairs)
        
        # Generate embeddings
        pred_embeddings = self.embedding_model.encode(
            list(pred_texts), 
            show_progress_bar=False,
            convert_to_numpy=True
        )
        truth_embeddings = self.embedding_model.encode(
            list(truth_texts),
            show_progress_bar=False,
            convert_to_numpy=True
        )
        
        # Compute metrics
        cosine_sims = [
            1 - cosine(pred_emb, truth_emb)
            for pred_emb, truth_emb in zip(pred_embeddings, truth_embeddings)
        ]
        
        euclidean_dists = [
            np.linalg.norm(pred_emb - truth_emb)
            for pred_emb, truth_emb in zip(pred_embeddings, truth_embeddings)
        ]
        
        dot_products = [
            np.dot(pred_emb, truth_emb)
            for pred_emb, truth_emb in zip(pred_embeddings, truth_embeddings)
        ]
        
        return {
            'cosine_similarity': float(np.mean(cosine_sims)),
            'euclidean_distance': float(np.mean(euclidean_dists)),
            'dot_product': float(np.mean(dot_products)),
            'count': len(valid_pairs)
        }
    
    def evaluate_activities(
        self, 
        predictions: List[str], 
        ground_truths: List[str]
    ) -> Dict:
        """Evaluates activity predictions with lexical and semantic metrics.
        
        Args:
            predictions: List of predicted activity descriptions.
            ground_truths: List of ground truth descriptions.
            
        Returns:
            Dictionary with all activity metrics.
        """
        # Filter valid pairs
        valid_pairs = [
            (str(pred), str(truth))
            for pred, truth in zip(predictions, ground_truths)
            if not pd.isna(pred) and not pd.isna(truth)
        ]
        
        if not valid_pairs:
            return {
                'bleu': 0.0,
                'rouge1': 0.0,
                'rouge2': 0.0,
                'rougeL': 0.0,
                'word_overlap': 0.0,
                'cosine_similarity': 0.0,
                'euclidean_distance': 0.0,
                'dot_product': 0.0,
                'count': 0
            }
        
        pred_texts, truth_texts = zip(*valid_pairs)
        
        # Lexical metrics
        bleu_scores = []
        rouge_scores = []
        word_overlaps = []
        
        for pred, truth in valid_pairs:
            bleu_scores.append(self._compute_bleu(pred, truth))
            rouge_scores.append(self._compute_rouge(pred, truth))
            word_overlaps.append(self._compute_word_overlap(pred, truth))
        
        # Average ROUGE scores
        avg_rouge = {
            'rouge1': np.mean([s['rouge1'] for s in rouge_scores]),
            'rouge2': np.mean([s['rouge2'] for s in rouge_scores]),
            'rougeL': np.mean([s['rougeL'] for s in rouge_scores])
        }
        
        # Semantic similarity metrics
        semantic_metrics = self._compute_semantic_similarity(
            list(pred_texts), 
            list(truth_texts)
        )
        
        return {
            # Lexical metrics
            'bleu': float(np.mean(bleu_scores)),
            'rouge1': avg_rouge['rouge1'],
            'rouge2': avg_rouge['rouge2'],
            'rougeL': avg_rouge['rougeL'],
            'word_overlap': float(np.mean(word_overlaps)),
            # Semantic metrics
            'cosine_similarity': semantic_metrics['cosine_similarity'],
            'euclidean_distance': semantic_metrics['euclidean_distance'],
            'dot_product': semantic_metrics['dot_product'],
            'count': len(valid_pairs)
        }

def evaluate_free_text_field(merged_df, field_name, evaluator):
    """Evaluate a free-text field using lexical and semantic metrics."""
    mask = (merged_df["_merge"] == "both") & merged_df[f"{field_name}_gt"].notna()
    
    predictions = merged_df.loc[mask, f"{field_name}_pred"].tolist()
    ground_truths = merged_df.loc[mask, f"{field_name}_gt"].tolist()
    
    metrics = evaluator.evaluate_activities(predictions, ground_truths)
    
    return pd.Series(metrics, name=field_name)
    

## Annotations part 1

In [6]:
GT_COLS = [
    "Context", "Location", "Activity", "Child_of_interest_clear",
    "#_adults", "#_children", "#_people_background",
    "Interaction_with_child", "#_people_interacting",
    "Child_constrained", "Constraint_type",
    "Supports", "Support_type", "Example_support_type"
]

# Map prediction keys -> ground_truth keys where they differ
PRED_TO_GT = {
    "Num_adults_foreground": "#_adults",
    "Num_children_foreground": "#_children",
    "Num_people_background": "#_people_background",
    "Num_people_interacting_with_child": "#_people_interacting",
}

NUM_COLS = ["#_adults", "#_children", "#_people_background", "#_people_interacting"]
TEXT_COLS = [c for c in GT_COLS if c not in NUM_COLS]

NA_STRINGS = {"n/a", "na", "none", "null", ""}
F1_COLS = [
    "Interaction_with_child",
    "Child_constrained",
    "Supports","Child_of_interest_clear"

]


In [7]:
import pandas as pd
from typing import Tuple

def compare_two_runs_metrics(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    *,
    key: str = "field",
    ddof: int = 0,
    rate_metrics: Tuple[str, ...] = (
        "accuracy","balanced_accuracy", "macro_f1", "weighted_f1",
        "precision", "recall", "sensitivity", "specificity",
    ),
    make_pm: bool = True,
) -> pd.DataFrame:

    def _ensure_key(d: pd.DataFrame) -> pd.DataFrame:
        d = d.copy()
        if key in d.columns:
            return d
        d = d.reset_index()
        if key not in d.columns:
            d = d.rename(columns={d.columns[0]: key})
        return d

    a = _ensure_key(df1)
    b = _ensure_key(df2)

    common_cols = [c for c in a.columns if c in b.columns and c != key]
    a = a[[key] + common_cols].copy()
    b = b[[key] + common_cols].copy()

    a = a.rename(columns={c: f"{c}_run1" for c in common_cols})
    b = b.rename(columns={c: f"{c}_run2" for c in common_cols})

    out = a.merge(b, on=key, how="outer")

    # ✅ (1) Drop “one-sided” rows (only run1 OR only run2 present)
    run1_cols = [f"{c}_run1" for c in common_cols]
    run2_cols = [f"{c}_run2" for c in common_cols]
    out = out[out[run1_cols].notna().any(axis=1) & out[run2_cols].notna().any(axis=1)].copy()

    for c in common_cols:
        c1 = f"{c}_run1"
        c2 = f"{c}_run2"

        out[f"{c}_mean"]  = out[[c1, c2]].mean(axis=1)
        out[f"{c}_std"]   = out[[c1, c2]].std(axis=1, ddof=ddof)
        out[f"{c}_delta"] = out[c2] - out[c1]

        if make_pm:
            if c in rate_metrics:
                out[f"{c}_pm"] = (
                    (100 * out[f"{c}_mean"]).round(2).astype("string")
                    + "% ± "
                    + (100 * out[f"{c}_std"]).round(2).astype("string")
                    + "%"
                )
            else:
                out[f"{c}_pm"] = (
                    out[f"{c}_mean"].round(2).astype("string")
                    + " ± "
                    + out[f"{c}_std"].round(2).astype("string")
                )

    # ✅ (2) If duplicates still exist per field, keep the “most different” comparison row
    if out.duplicated(subset=[key]).any():
        if "accuracy_delta" in out.columns:
            score = out["accuracy_delta"].abs()
        else:
            delta_cols = [f"{c}_delta" for c in common_cols]
            score = out[delta_cols].abs().sum(axis=1, min_count=1)
        out["_score"] = score.fillna(0)

        out = (out.sort_values([key, "_score"], ascending=[True, False])
                 .drop_duplicates(subset=[key], keep="first")
                 .drop(columns="_score"))

    if "accuracy_mean" in out.columns:
        out = out.sort_values("accuracy_mean", ascending=False, na_position="last")

    return out.reset_index(drop=True)


In [8]:
def compare_accuracy_two_runs(df1, df2, *, key="annotation", acc_col="accuracy", suffixes=("_run1", "_run2")):
    """
    Returns a dataframe with per-annotation:
      - accuracy_run1, accuracy_run2
      - accuracy_mean = (a1 + a2)/2
      - accuracy_std  = std([a1, a2], ddof=0)  # population std -> 0.54/0.44 => 0.05
      - accuracy_pm   = pretty string like '49.0% ± 5.0%'
      - delta         = a2 - a1
    """
    a = df1[[key, acc_col, "n_eval", "n_correct"]].copy()
    b = df2[[key, acc_col, "n_eval", "n_correct"]].copy()

    a = a.rename(columns={acc_col: f"{acc_col}{suffixes[0]}", "n_eval": f"n_eval{suffixes[0]}", "n_correct": f"n_correct{suffixes[0]}"})
    b = b.rename(columns={acc_col: f"{acc_col}{suffixes[1]}", "n_eval": f"n_eval{suffixes[1]}", "n_correct": f"n_correct{suffixes[1]}"})

    out = a.merge(b, on=key, how="outer")

    col1 = f"{acc_col}{suffixes[0]}"
    col2 = f"{acc_col}{suffixes[1]}"

    # mean across runs
    out["accuracy_mean"] = out[[col1, col2]].mean(axis=1)

    # population std across runs (ddof=0)
    out["accuracy_std"] = out[[col1, col2]].std(axis=1, ddof=0)

    # signed change
    out["delta"] = out[col2] - out[col1]

    # pretty formatting
    out["accuracy_pm"] = (
        (100 * out["accuracy_mean"]).round(2).astype("string")
        + "% ± "
        + (100 * out["accuracy_std"]).round(2).astype("string")
        + "%"
    )

    # optional: sort by mean accuracy desc
    out = out.sort_values("accuracy_mean", ascending=False, na_position="last").reset_index(drop=True)
    return out

In [9]:
print("Prediction Evaluation for the first split of annotations with Ovis2\n")
pred_1_ovis,gt_1_ovis,merged_1_ovis, metrics_1_ovis=analyse_df(results_1_ovis,GT_COLS,PRED_TO_GT,TEXT_COLS,NUM_COLS)
pred_1_2_ovis,gt_1_2_ovis,merged_1_2_ovis, metrics_1_2_ovis=analyse_df(results_1_2_ovis,GT_COLS,PRED_TO_GT,TEXT_COLS,NUM_COLS)
summary=compare_accuracy_two_runs(metrics_1_ovis,metrics_1_2_ovis)
print(summary[[ "annotation", "accuracy_run1", "accuracy_run2", "accuracy_pm", "delta" ]])
metrics = []
for c in F1_COLS:
    m = compute_f1_metrics(merged_1_ovis, c)
    metrics.append({"field": c, **m})

metrics_df_1_ovis_f1 = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
for c in F1_COLS:
    m = compute_f1_metrics(merged_1_2_ovis, c)
    metrics.append({"field": c, **m})

metrics_df_1_2_ovis_f1 = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
summary = compare_two_runs_metrics(metrics_df_1_ovis_f1, metrics_df_1_2_ovis_f1, key="field")

# show the “balanced ± std” view for the rate metrics you care about
cols = ["field", "accuracy_pm", "macro_f1_pm", "weighted_f1_pm", "precision_pm", "recall_pm", "specificity_pm"]
print(summary[cols])
# Evaluate Activity and Context
evaluator = Evaluator(embedding_model_name='all-MiniLM-L6-v2')
activity_metrics_ovis = evaluate_free_text_field(merged_1_ovis, "Activity", evaluator)
context_metrics_ovis = evaluate_free_text_field(merged_1_ovis, "Context", evaluator)

print(activity_metrics_ovis)
print(context_metrics_ovis)


Prediction Evaluation for the first split of annotations with Ovis2

{'Num_adults_foreground': '#_adults', 'Num_children_foreground': '#_children', 'Num_people_background': '#_people_background', 'Num_people_interacting_with_child': '#_people_interacting'}
{'Num_adults_foreground': '#_adults', 'Num_children_foreground': '#_children', 'Num_people_background': '#_people_background', 'Num_people_interacting_with_child': '#_people_interacting'}
                 annotation  accuracy_run1  accuracy_run2     accuracy_pm  \
0                  Location       0.884766       0.885370  88.51% ± 0.03%   
1   Child_of_interest_clear       0.858135       0.857833   85.8% ± 0.02%   
2       #_people_background       0.857230       0.853281   85.53% ± 0.2%   
3                #_children       0.840724       0.842232  84.15% ± 0.08%   
4                  #_adults       0.833183       0.833786  83.35% ± 0.03%   
5         Child_constrained       0.808992       0.810501  80.97% ± 0.08%   
6               

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 57877651-fe60-4a6c-9520-24732690f805)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 8867f550-066d-458b-ae16-c119e8bdee29)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json
Retrying in 1s [Retry 1/5].


bleu                     0.040048
rouge1                   0.186642
rouge2                   0.051328
rougeL                   0.177841
word_overlap             0.206044
cosine_similarity        0.384589
euclidean_distance       1.090073
dot_product              0.384589
count                 3314.000000
Name: Activity, dtype: float64
bleu                     0.282174
rouge1                   0.536652
rouge2                   0.491704
rougeL                   0.536652
word_overlap             0.541026
cosine_similarity        0.611124
euclidean_distance       0.622003
dot_product              0.611124
count                 3315.000000
Name: Context, dtype: float64


In [10]:
print("Prediction Evaluation for the first split of annotations with Qwen2.5\n")
pred_1_qwen,gt_1_qwen,merged_1_qwen,metrics_1_qwen=analyse_df(results_1_qwen,GT_COLS,PRED_TO_GT,TEXT_COLS,NUM_COLS)
pred_1_2_qwen,gt_1_2_qwen,merged_1_2_qwen,metrics_1_2_qwen=analyse_df(results_1_2_qwen,GT_COLS,PRED_TO_GT,TEXT_COLS,NUM_COLS)
summary=compare_accuracy_two_runs(metrics_1_qwen,metrics_1_2_qwen)
print(summary[[ "annotation", "accuracy_run1", "accuracy_run2", "accuracy_pm", "delta" ]])
metrics = []
for c in F1_COLS:
    m = compute_f1_metrics(merged_1_qwen, c)
    metrics.append({"field": c, **m})

metrics_df_1_ovis_f1 = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
for c in F1_COLS:
    m = compute_f1_metrics(merged_1_2_qwen, c)
    metrics.append({"field": c, **m})

metrics_df_1_2_ovis_f1 = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
summary = compare_two_runs_metrics(metrics_df_1_ovis_f1, metrics_df_1_2_ovis_f1, key="field")

# show the “balanced ± std” view for the rate metrics you care about
cols = ["field", "accuracy_pm", "macro_f1_pm", "weighted_f1_pm", "precision_pm", "recall_pm", "specificity_pm"]
print(summary[cols])
activity_metrics_qwen = evaluate_free_text_field(merged_1_qwen, "Activity", evaluator)
context_metrics_qwen = evaluate_free_text_field(merged_1_qwen, "Context", evaluator)
print(activity_metrics_qwen)
print(context_metrics_qwen)

Prediction Evaluation for the first split of annotations with Qwen2.5

{'Num_adults_foreground': '#_adults', 'Num_children_foreground': '#_children', 'Num_people_background': '#_people_background', 'Num_people_interacting_with_child': '#_people_interacting'}
{'Num_adults_foreground': '#_adults', 'Num_children_foreground': '#_children', 'Num_people_background': '#_people_background', 'Num_people_interacting_with_child': '#_people_interacting'}
                 annotation  accuracy_run1  accuracy_run2     accuracy_pm  \
0   Child_of_interest_clear       0.891639       0.894356   89.3% ± 0.14%   
1       #_people_background       0.856926       0.856926   85.69% ± 0.0%   
2                  Location       0.853695       0.854299   85.4% ± 0.03%   
3                #_children       0.852790       0.853997  85.34% ± 0.06%   
4         Child_constrained       0.840374       0.842788  84.16% ± 0.12%   
5                  #_adults       0.838612       0.837707  83.82% ± 0.05%   
6             

## Annotations part 2

In [11]:
##### -----------------------
GT2_COLS = [
    "Gestures", "Gesture_type", "Vocalizations",
    "RMM", "RMM_type", "Response_to_name",
    "Locomotion", "Locomotion_type",
    "Grasping", "Grasp_type"
]

F1_2_COLS = [
    "Gestures", "Vocalizations",
    "RMM",  "Response_to_name",
    "Locomotion", 
    "Grasping", 
]
print("Prediction Evaluation for the second split of annotations with Ovis2\n")
pred_2_ovis,gt_2_ovis,merged_2_ovis,metrics_2_ovis=analyse_df(results_2_ovis,GT_COLS=GT2_COLS)
pred_2_2_ovis,gt_2_2_ovis,merged_2_2_ovis,metrics_2_2_ovis=analyse_df(results_2_2_ovis,GT2_COLS)
summary=compare_accuracy_two_runs(metrics_2_ovis,metrics_2_2_ovis)
print(summary[[ "annotation", "accuracy_run1", "accuracy_run2", "accuracy_pm", "delta" ]])
metrics = []
for c in F1_2_COLS:
    m = compute_f1_metrics(merged_2_ovis, c)
    metrics.append({"field": c, **m})

metrics_df_1_ovis_f1 = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
for c in F1_2_COLS:
    m = compute_f1_metrics(merged_2_2_ovis, c)
    metrics.append({"field": c, **m})

metrics_df_1_2_ovis_f1 = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
summary = compare_two_runs_metrics(metrics_df_1_ovis_f1, metrics_df_1_2_ovis_f1, key="field")

# show the “balanced ± std” view for the rate metrics you care about
cols = ["field", "accuracy_pm", "macro_f1_pm", "weighted_f1_pm", "precision_pm", "recall_pm", "specificity_pm"]
print(summary[cols])
metrics = []
for c in F1_2_COLS:
    m = compute_f1_metrics(merged_2_ovis, c)
    metrics.append({"field": c, **m})

metrics_df_2_ovis= pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
metrics_df_2_ovis

Prediction Evaluation for the second split of annotations with Ovis2

{}
{}
         annotation  accuracy_run1  accuracy_run2     accuracy_pm     delta
0               RMM       0.842788       0.843693  84.32% ± 0.05%  0.000905
1          Grasping       0.668075       0.669584  66.88% ± 0.08%  0.001509
2        Locomotion       0.622471       0.620356  62.14% ± 0.11% -0.002114
3          Gestures       0.584365       0.584968  58.47% ± 0.03%  0.000604
4        Grasp_type       0.507829       0.521921   51.49% ± 0.7%  0.014092
5     Vocalizations       0.264937       0.265842  26.54% ± 0.05%  0.000905
6      Gesture_type       0.212197       0.219664  21.59% ± 0.37%  0.007467
7   Locomotion_type       0.093662       0.098592   9.61% ± 0.25%  0.004930
8          RMM_type       0.040323       0.043011   4.17% ± 0.13%  0.002688
9  Response_to_name       0.013216       0.013216    1.32% ± 0.0%  0.000000
              field     accuracy_pm     macro_f1_pm  weighted_f1_pm  \
0               R

,n,accuracy,macro_f1,weighted_f1,precision,recall,sensitivity,specificity,tp,fp,tn,fn
field,,,,,,,,,,,,
RMM,3314,0.842788,0.551342,0.831768,0.225092,0.163978,0.163978,0.928620,61,210,2732,311
Grasping,3314,0.668075,0.660902,0.668402,0.714361,0.706129,0.706129,0.616370,1348,539,866,561
Gestures,3313,0.584365,0.584146,0.584477,0.567766,0.581614,0.581614,0.586931,930,708,1006,669
Locomotion,3311,0.622471,0.405363,0.536408,0.925926,0.143062,0.143062,0.991636,200,16,1897,1198
Vocalizations,3314,0.264937,0.216749,0.122483,0.903226,0.011377,0.011377,0.996483,28,3,850,2433
Response_to_name,227,0.013216,0.013636,0.025230,NaN,0.000000,0.000000,1.000000,0,0,129,98


In [12]:
print("Prediction Evaluation for the second split of annotations with Qwen2.5\n")

pred_2_qwen,gt_2_qwen,merged_2_qwen,metrics_2_qwen=analyse_df(results_2_qwen,GT_COLS=GT2_COLS)
pred_2_2_qwen,gt_2_2_qwen,merged_2_2_qwen,metrics_2_2_qwen=analyse_df(results_2_2_qwen,GT2_COLS)
summary=compare_accuracy_two_runs(metrics_2_qwen,metrics_2_2_qwen)
print(summary[[ "annotation", "accuracy_run1", "accuracy_run2", "accuracy_pm", "delta" ]])
metrics = []
for c in F1_2_COLS:
    m = compute_f1_metrics(merged_2_qwen, c)
    metrics.append({"field": c, **m})

metrics_df_1_ovis_f1 = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
for c in F1_2_COLS:
    m = compute_f1_metrics(merged_2_2_qwen, c)
    metrics.append({"field": c, **m})

metrics_df_1_2_ovis_f1 = pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
summary = compare_two_runs_metrics(metrics_df_1_ovis_f1, metrics_df_1_2_ovis_f1, key="field")

# show the “balanced ± std” view for the rate metrics you care about
cols = ["field", "accuracy_pm", "macro_f1_pm", "weighted_f1_pm", "precision_pm", "recall_pm", "specificity_pm"]
print(summary[cols])
metrics = []

for c in F1_2_COLS:
    m = compute_f1_metrics(merged_2_qwen, c)
    metrics.append({"field": c, **m})

metrics_df_qwen= pd.DataFrame(metrics).set_index("field").sort_values("weighted_f1", ascending=False)
metrics_df_qwen

Prediction Evaluation for the second split of annotations with Qwen2.5

{}
{}
         annotation  accuracy_run1  accuracy_run2     accuracy_pm     delta
0               RMM       0.884731       0.884128  88.44% ± 0.03% -0.000604
1        Locomotion       0.599215       0.596195  59.77% ± 0.15% -0.003020
2          Gestures       0.554482       0.555388  55.49% ± 0.05%  0.000906
3          Grasping       0.471032       0.480688  47.59% ± 0.48%  0.009656
4     Vocalizations       0.347013       0.349427  34.82% ± 0.12%  0.002414
5      Gesture_type       0.076540       0.090230   8.34% ± 0.68%  0.013690
6        Grasp_type       0.069415       0.088205   7.88% ± 0.94%  0.018789
7   Locomotion_type       0.048592       0.037324    4.3% ± 0.56% -0.011268
8  Response_to_name       0.017621       0.013216   1.54% ± 0.22% -0.004405
9          RMM_type       0.000000       0.000000     0.0% ± 0.0%  0.000000
              field     accuracy_pm     macro_f1_pm  weighted_f1_pm  \
0              

,n,accuracy,macro_f1,weighted_f1,precision,recall,sensitivity,specificity,tp,fp,tn,fn
field,,,,,,,,,,,,
RMM,3314,0.884731,0.489442,0.837828,0.307692,0.021505,0.021505,0.993882,8,18,2924,364
Gestures,3313,0.554482,0.497574,0.503443,0.602671,0.225766,0.225766,0.861144,361,238,1476,1238
Locomotion,3311,0.599215,0.291537,0.479661,0.886957,0.072961,0.072961,0.993204,102,13,1900,1296
Grasping,3314,0.471032,0.390749,0.357115,0.886139,0.093766,0.093766,0.983630,179,23,1382,1730
Vocalizations,3314,0.347013,0.333492,0.287432,0.889764,0.137749,0.137749,0.950762,339,42,811,2122
Response_to_name,227,0.017621,0.018393,0.033414,0.500000,0.010204,0.010204,0.992248,1,1,128,97


In [13]:
len(gt_2_qwen[gt_2_qwen["Vocalizations"]=="yes"])/len(gt_2_qwen[~gt_2_qwen["Vocalizations"].isna()])

0.7426071213035607

In [14]:
set(pred_2_qwen["Locomotion_type"])

{<NA>, 'crawl', 'cruise', 'run', 'vehicle', 'walk'}

## Annotations part 3

In [15]:
GT3_TEXT_COLS = ["Body_Parts_Visible", "Angle_of_Body"]
GT3_NUM_COLS = [
    "Video_Quality_Child_Face_Visibility",
    "Video_Quality_Child_Body_Visibility",
    "Video_Quality_Child_Hand_Visibility",
    "Video_Quality_Lighting",
    "Video_Quality_Resolution",
    "Video_Quality_Motion",
]
GT3_COLS = GT3_TEXT_COLS + GT3_NUM_COLS


In [16]:
print("Prediction Evaluation for the third split of annotations with Ovis2\n")
pred_3_ovis,gt_3_ovis,merged_3_ovis,metrics_3_ovis=analyse_df(results_3_ovis,GT_COLS=GT3_COLS,TEXT_COLS=GT3_TEXT_COLS,NUM_COLS=GT3_NUM_COLS)
pred_3_2_ovis,gt_3_2_ovis,merged_3_2_ovis,metrics_3_2_ovis=analyse_df(results_3_2_ovis,GT_COLS=GT3_COLS,TEXT_COLS=GT3_TEXT_COLS,NUM_COLS=GT3_NUM_COLS)
summary=compare_accuracy_two_runs(metrics_3_ovis,metrics_3_2_ovis)

print(summary[[ "annotation", "accuracy_run1", "accuracy_run2", "accuracy_pm", "delta" ]])


Prediction Evaluation for the third split of annotations with Ovis2

{}
{}
                            annotation  accuracy_run1  accuracy_run2  \
0                   Body_Parts_Visible       0.763214       0.754455   
1                        Angle_of_Body       0.405316       0.418001   
2  Video_Quality_Child_Body_Visibility       0.212798       0.218835   
3  Video_Quality_Child_Face_Visibility       0.149366       0.153893   
4  Video_Quality_Child_Hand_Visibility       0.148159       0.148159   
5                 Video_Quality_Motion       0.054315       0.053410   
6               Video_Quality_Lighting       0.018105       0.019916   
7             Video_Quality_Resolution       0.002414       0.002716   

      accuracy_pm     delta  
0  75.88% ± 0.44% -0.008759  
1  41.17% ± 0.63%  0.012685  
2   21.58% ± 0.3%  0.006037  
3  15.16% ± 0.23%  0.004526  
4   14.82% ± 0.0%  0.000000  
5   5.39% ± 0.05% -0.000905  
6    1.9% ± 0.09%  0.001811  
7   0.26% ± 0.02%  0.000302  


In [17]:
print("Prediction Evaluation for the third split of annotations with Qwen2.5\n")
pred_3_qwen,gt_3_qwen,merged_3_qwen,metrics_3_qwen=analyse_df(results_3_qwen,GT_COLS=GT3_COLS,TEXT_COLS=GT3_TEXT_COLS,NUM_COLS=GT3_NUM_COLS)
pred_3_2_qwen,gt_3_2_qwen,merged_2_2_qwen,metrics_3_2_qwen=analyse_df(results_3_2_qwen,GT_COLS=GT3_COLS,TEXT_COLS=GT3_TEXT_COLS,NUM_COLS=GT3_NUM_COLS)
summary=compare_accuracy_two_runs(metrics_3_qwen,metrics_3_2_qwen)
print(summary[[ "annotation", "accuracy_run1", "accuracy_run2", "accuracy_pm", "delta" ]])


Prediction Evaluation for the third split of annotations with Qwen2.5

{}
{}
                            annotation  accuracy_run1  accuracy_run2  \
0                   Body_Parts_Visible       0.623679       0.609786   
1                        Angle_of_Body       0.289943       0.288130   
2  Video_Quality_Child_Body_Visibility       0.233927       0.222759   
3  Video_Quality_Child_Face_Visibility       0.210320       0.208208   
4  Video_Quality_Child_Hand_Visibility       0.138202       0.131865   
5               Video_Quality_Lighting       0.089016       0.100483   
6             Video_Quality_Resolution       0.090525       0.096862   
7                 Video_Quality_Motion       0.062764       0.054013   

      accuracy_pm     delta  
0  61.67% ± 0.69% -0.013893  
1   28.9% ± 0.09% -0.001812  
2  22.83% ± 0.56% -0.011168  
3  20.93% ± 0.11% -0.002112  
4   13.5% ± 0.32% -0.006337  
5   9.47% ± 0.57%  0.011467  
6   9.37% ± 0.32%  0.006337  
7   5.84% ± 0.44% -0.008751  
